In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
# Fully Connected Model Definition
class FullyConnectedModel(nn.Module):
    def __init__(self):
        super(FullyConnectedModel, self).__init__()
        self.fc1 = nn.Linear(128*128*3, 2816)
        self.fc2 = nn.Linear(2816, 2560)
        self.fc3 = nn.Linear(2560, 2304)
        self.fc4 = nn.Linear(2304, 2048)
        self.fc5 = nn.Linear(2048, 1792)
        self.fc6 = nn.Linear(1792, 1536)
        self.fc7 = nn.Linear(1536, 1280)
        self.fc8 = nn.Linear(1280, 1024)
        self.fc9 = nn.Linear(1024, 768)
        self.fc10 = nn.Linear(768, 512)
        self.fc11 = nn.Linear(512, 256)
        self.fc12 = nn.Linear(256, 128)
        self.fc13 = nn.Linear(128, 64)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        x = torch.relu(self.fc6(x))
        x = torch.relu(self.fc7(x))
        x = torch.relu(self.fc8(x))
        x = torch.relu(self.fc9(x))
        x = torch.relu(self.fc10(x))
        x = torch.relu(self.fc11(x))
        x = torch.relu(self.fc12(x))
        x = self.fc13(x)
        return x

In [4]:
# Convolutional Neural Network (CNN) Definition
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout(0.25)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout(0.25)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop3 = nn.Dropout(0.25)

        self.conv4 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)

        self.fc1 = nn.Linear(64 * 16 * 16, 64)  # Assuming input size is 128x128
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.drop1(x)

        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.drop2(x)

        x = self.pool3(torch.relu(self.conv3(x)))
        x = self.drop3(x)

        x = torch.relu(self.conv4(x))

        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return x

In [5]:
# Hybrid Model Definition
class HybridModel(nn.Module):
    def __init__(self):
        super(HybridModel, self).__init__()
        self.fc_model = FullyConnectedModel()
        self.cnn_model = CNNModel()
        self.fc_final = nn.Linear(128, 10)

    def forward(self, visual_input, trajectory_input):
        visual_output = self.cnn_model(visual_input)
        trajectory_output = self.fc_model(trajectory_input)

        combined = torch.cat((visual_output, trajectory_output), dim=1)
        output = self.fc_final(combined)
        return output

In [6]:
# Hyperparameters
num_epochs = 5
batch_size = 32
learning_rate = 0.001

In [7]:
# Example dataset (placeholders for actual visual and trajectory data)
visual_data = torch.randn(500, 3, 128, 128)  # 500 samples of 128x128 RGB images
trajectory_data = torch.randn(500, 128*128*3)  # Corresponding trajectory weights
labels = torch.randn(500, 10)  # Target labels for the output

In [8]:
# Dataset and DataLoader
dataset = torch.utils.data.TensorDataset(visual_data, trajectory_data, labels)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [9]:
# Initialize the model, criterion, and optimizer
model = HybridModel()
criterion = nn.MSELoss()  # Using MSE loss as mentioned in the paper for trajectory prediction
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [10]:
# Training loop
total_step = len(train_loader)
for epoch in range(num_epochs):
    for i, (visuals, trajectories, targets) in enumerate(train_loader):
        # Forward pass
        outputs = model(visuals, trajectories)
        loss = criterion(outputs, targets)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{total_step}], Loss: {loss.item():.4f}')

print('Finished Training')

Epoch [1/5], Step [10/16], Loss: 1.0527
Epoch [2/5], Step [10/16], Loss: 0.9911
Epoch [3/5], Step [10/16], Loss: 1.0619
Epoch [4/5], Step [10/16], Loss: 0.9878
Epoch [5/5], Step [10/16], Loss: 0.9813
Finished Training
